# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드
- 저장 컬럼: `id`, `tags`, `file_url`, `sample_url`, `width`, `height`
- 중단 후 이어서 크롤링 가능

In [ ]:
import os
import requests
import queue
import threading
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from fake_useragent import UserAgent

# --- 사용자 설정 ---
# 파일이 있으면 파일 내 최저 ID부터 시작하고, 없으면 이 값을 사용합니다.
INITIAL_START_ID = 6654626 
END_ID = 1
NUM_THREADS = 10 
SAVE_INTERVAL = 100
SAVE_PATH = r"../data/metadata_html.parquet"

# --- 전역 변수 및 락 ---
buffer = []
save_lock = threading.Lock()
task_queue = queue.Queue(maxsize=2000)
ua = UserAgent()

def get_last_processed_id():
    """기존 파켓 파일을 읽어 다음에 시작할 ID를 결정합니다."""
    if os.path.exists(SAVE_PATH):
        try:
            df = pd.read_parquet(SAVE_PATH)
            if not df.empty:
                # 역순(START -> END) 수집이므로 가장 작은 ID를 찾아 그 아래부터 시작
                last_id = df['id'].min()
                print(f"기존 데이터를 발견했습니다. 마지막 ID {last_id} 다음인 {last_id - 1}부터 시작합니다.")
                return last_id - 1
        except Exception as e:
            print(f"기존 파일 읽기 실패(새로 시작): {e}")
    return INITIAL_START_ID

def fetch_api_data(post_id):
    url = f"https://safebooru.org/index.php?page=dapi&s=post&q=index&id={post_id}"
    headers = {'User-Agent': ua.random}
    
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            post = root.find('post')
            if post is not None:
                return {
                    "id": int(post.get('id')),
                    "tags": post.get('tags'),
                    "file_url": post.get('file_url') if post.get('file_url').startswith('http') else "https:" + post.get('file_url'),
                    "width": int(post.get('width')),
                    "height": int(post.get('height'))
                }
        return None
    except:
        return None

def save_buffer(pbar=None):
    global buffer
    
    # 1. 락을 먼저 걸어 스레드 간 동기화 보장
    with save_lock:
        # 2. 리스트 길이를 명시적으로 체크
        if len(buffer) == 0:
            return
            
        try:
            # 새로운 데이터 생성 후 버퍼 비우기
            new_df = pd.DataFrame(buffer)
            buffer = [] 
            
            os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
            
            # 3. 파일 합치기 로직
            if os.path.exists(SAVE_PATH):
                try:
                    existing_df = pd.read_parquet(SAVE_PATH)
                    # 데이터프레임 존재 여부를 len()으로 체크하여 TypeError 방지
                    if existing_df is not None and len(existing_df) > 0:
                        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
                        combined_df = combined_df.drop_duplicates(subset=['id'])
                    else:
                        combined_df = new_df
                except Exception as e:
                    if pbar: pbar.write(f"파일 읽기 중 오류: {e}")
                    combined_df = new_df
            else:
                combined_df = new_df

            # 4. 최종 저장 전 검사 및 실행
            if combined_df is not None and len(combined_df) > 0:
                combined_df.to_parquet(SAVE_PATH, index=False)
                if pbar:
                    pbar.set_postfix({"누적저장": f"{len(combined_df)}건"})
                    
        except Exception as e:
            if pbar:
                pbar.write(f"저장 로직 내부 오류: {e}")

def worker(pbar):
    while True:
        post_id = task_queue.get()
        if post_id is None:
            task_queue.task_done()
            break
        
        result_data = fetch_api_data(post_id)
        if result_data:
            with save_lock:
                buffer.append(result_data)
                if len(buffer) >= SAVE_INTERVAL:
                    save_buffer(pbar)
        
        pbar.update(1)
        task_queue.task_done()

if __name__ == "__main__":
    # 시작 ID 자동 결정
    current_start_id = get_last_processed_id()
    
    print(f"=== 고속 수집기 가동 (ID: {current_start_id} ~ {END_ID}) ===")
    
    total_tasks = current_start_id - END_ID + 1
    pbar = tqdm(total=total_tasks, desc="수집 진행률", ncols=100)
    
    threads = []
    for _ in range(NUM_THREADS):
        t = threading.Thread(target=worker, args=(pbar,))
        t.daemon = True
        t.start()
        threads.append(t)
        
    try:
        # 결정된 시작 ID부터 END_ID까지 역순으로 큐에 삽입
        for pid in range(current_start_id, END_ID - 1, -1):
            task_queue.put(pid)
        task_queue.join()
    except KeyboardInterrupt:
        print("\n[!] 중단 요청 감지. 현재 버퍼를 저장하고 안전하게 종료합니다.")
    
    for _ in range(NUM_THREADS):
        task_queue.put(None)
    for t in threads:
        t.join()
        
    save_buffer(pbar)
    pbar.close()
    print("=== 모든 작업 완료 ===")

기존 데이터를 발견했습니다. 마지막 ID 6524346 다음인 6524345부터 시작합니다.
=== 고속 수집기 가동 (ID: 6524345 ~ 1) ===


수집 진행률:   0%|                                        | 282/6524345 [00:19<121:15:40, 14.94it/s]Exception in thread Thread-5 (worker):
Traceback (most recent call last):
  File "C:\Users\EL069\AppData\Local\Temp\ipykernel_19388\2199783464.py", line 86, in save_buffer
TypeError: __bool__ should return bool, returned numpy.bool

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\EL069\anaconda3\envs\safebooru\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\EL069\anaconda3\envs\safebooru\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\EL069\AppData\Local\Temp\ipykernel_19388\2199783464.py", line 108, in worker
  File "C:\Users\EL069\AppData\Local\Temp\ipykernel_19388\2199783464.py", line 90, in save_buffer
TypeError: __bool__ should return bool, returned numpy.bool
수집 진행률:   0%|                                        | 284/6524345 [00:19